# Framework Comparison — 3 Modes

Same model (MiniMax M2.5), same prompt, same tools, same trace.  
1. **DSPy ReAct (text-based)** — `react_native_fc.ReAct(use_native_function_calling=False)` + XMLAdapter
2. **DSPy ReAct (native FC)** — `react_native_fc.ReAct(use_native_function_calling=True)`
3. **PydanticAI Agent (native FC)** — different framework, native FC by default

In [ ]:
import sys, os, tempfile, shutil
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../.env"))
load_dotenv(Path("../../lerim-cloud/.env"))

sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("../../lerim-cloud").resolve()))

# ── FLAGS ──
RUN_DSPY_TEXT = False                # ReAct with use_native_function_calling=False (text-based)  — legacy
RUN_DSPY_NATIVE_FC = False           # ReAct with use_native_function_calling=True (native FC)     — legacy
RUN_PYDANTICAI_SINGLE_PASS = True    # PydanticAI Agent, one pass (extract_pydanticai.run_extraction)
RUN_PYDANTICAI_THREE_PASS = True     # PydanticAI Agent, three passes (extract.run_extraction_three_pass)

# ── MODEL: switch between "minimax" and "ollama" ──
USE_MODEL = "gpt-oss-20b"  # "minimax" or "gpt-oss-20b" or "gemma4-e2b"

MODEL_CONFIGS = {
    "minimax": {
        "dspy_provider": "minimax",
        "dspy_model": "MiniMax-M2.5",
        "pai_provider": "minimax",
        "pai_model": "MiniMax-M2.5",
    },
    "gpt-oss-20b": {
        "dspy_provider": "ollama",
        "dspy_model": "gpt-oss:20b",
        "pai_provider": "ollama",
        "pai_model": "gpt-oss:20b",
    },
    "gemma4-e2b": {
        "dspy_provider": "ollama",
        "dspy_model": "gemma4:E2b",
        "pai_provider": "ollama",
        "pai_model": "gemma4:E2b",
    },
}
MODEL_CFG = MODEL_CONFIGS[USE_MODEL]

# Trace
TRACE = Path("/Users/kargarisaac/codes/personal/lerim/lerim-cloud/evals/data/traces/claude_003.jsonl")
assert TRACE.exists(), f"Trace not found: {TRACE}"
print(f"Trace: {TRACE.name} ({sum(1 for _ in open(TRACE))} lines)")
print(f"Model: {USE_MODEL} ({MODEL_CFG['pai_model']})")
print(f"Modes: 1pass={RUN_PYDANTICAI_SINGLE_PASS}, 3pass={RUN_PYDANTICAI_THREE_PASS}")

In [16]:
def make_temp_memory():
    """Create a fresh temp memory directory."""
    tmp = Path(tempfile.mkdtemp(prefix="fw_compare_"))
    mem = tmp / "memory"
    mem.mkdir(parents=True)
    (mem / "summaries").mkdir()
    (mem / "index.md").write_text("# Memory Index\n")
    return tmp, mem

def report(mem: Path):
    """Print what the agent wrote."""
    import frontmatter as fm_lib
    memories = [f for f in sorted(mem.glob("*.md")) if f.name != "index.md"]
    summaries = list(sorted((mem / "summaries").glob("*.md")))
    print(f"\nMemories: {len(memories)}")
    for m in memories:
        try:
            post = fm_lib.load(str(m))
            print(f"  [{post.get('type','')}] {m.name}")
            print(f"    {post.get('description','')[:100]}")
        except Exception as e:
            # Model wrote malformed YAML — show raw content instead of crashing
            print(f"  [YAML ERROR] {m.name} ({type(e).__name__})")
            print(f"    {m.read_text()[:200]}")
    print(f"Summaries: {len(summaries)}")
    for s in summaries:
        print(f"  {s.name}")
    print(f"Index:\n{(mem / 'index.md').read_text()[:500]}")

## 1. PydanticAI Single-Pass (one agent, all six tools in one loop)

Baseline: one `Agent` with the combined extract+write+index+summary prompt. All tools (`read`, `grep`, `scan`, `write`, `edit`, `verify_index`) available to one LLM loop. Called via `lerim.agents.extract_pydanticai.run_extraction`.

In [ ]:
if RUN_PYDANTICAI_SINGLE_PASS:
    from lerim.agents.extract_pydanticai import build_extract_agent
    from lerim.agents.tools import ExtractDeps
    from lerim.config.providers import build_pydantic_model_from_provider

    tmp_pai, mem_pai = make_temp_memory()
    # Build a robust model via the canonical builder in lerim.config.providers.
    # Reads provider base URLs from default.toml [providers] and API keys from
    # env vars. Wraps httpx with AsyncTenacityTransport (HTTP retry for 429/
    # 5xx/network) — no hardcoded endpoints.
    model = build_pydantic_model_from_provider(
        MODEL_CFG["pai_provider"],
        MODEL_CFG["pai_model"],
    )
    agent = build_extract_agent(model)
    # New ExtractDeps lives in lerim.agents.tools (not extract_pydanticai).
    # It holds paths only — the tool functions resolve them via ctx.deps
    # without a MemoryTools instance.
    deps = ExtractDeps(memory_root=mem_pai, trace_path=TRACE)

    print(f"Running PydanticAI single-pass with {MODEL_CFG['pai_model']}...")
    result_pai = await agent.run("Extract memories from the session trace.", deps=deps)
    print(f"Summary: {result_pai.output.completion_summary}")
    report(mem_pai)

    # Conversation log
    print(f"\n{'=' * 70}")
    print(f"AGENT CONVERSATION LOG ({len(result_pai.all_messages())} messages)")
    print(f"{'=' * 70}")
    for i, msg in enumerate(result_pai.all_messages()):
        kind = msg.kind
        print(f"\n--- Message {i} [{kind}] ---")
        if kind == "request":
            for part in msg.parts:
                ptype = part.part_kind
                if ptype == "system-prompt":
                    print(f"  [system-prompt] ({len(part.content)} chars)")
                elif ptype == "user-prompt":
                    print(f"  [user-prompt] {part.content[:200]}")
                elif ptype == "tool-return":
                    print(f"  [tool-return] {part.tool_name} -> {str(part.content)[:200]}")
                else:
                    print(f"  [{ptype}] {str(part)[:200]}")
        elif kind == "response":
            for part in msg.parts:
                ptype = part.part_kind
                if ptype == "text":
                    print(f"  [text] {part.content[:300]}")
                elif ptype == "tool-call":
                    print(f"  [tool-call] {part.tool_name}({str(part.args)[:150]})")
                else:
                    print(f"  [{ptype}] {str(part)[:200]}")
else:
    print("SKIPPED (RUN_PYDANTICAI_SINGLE_PASS=False)")

## 2. PydanticAI Three-Pass (reflect → extract → finalize)

Three specialized agents chained in sequence:

1. **Reflect** — read-only tools (`read`, `grep`, `scan`); reads trace chunk-by-chunk and produces a structured `SessionUnderstanding` (user_goal, key_decisions, important_chunks, extractable_candidates, existing_memories_relevant). No writes.
2. **Extract** — receives the `SessionUnderstanding` as input; uses `read`, `grep`, `scan`, `write`, `edit` to dedup + write novel memories or update existing ones. Does not touch the index or summary.
3. **Finalize** — receives both the understanding and the list of filenames written by Pass 2; calls `verify_index`, fixes any mismatches via `edit`, and writes the session summary via `write(type="summary", ...)`.

Each pass has its own system prompt, `UsageLimits` (`request_limit` budget), structured `output_type`, and tool subset. Cells below run them inline so you can inspect each pass's conversation log separately.

**Empty-trace short-circuit:** after Pass 1, if the `SessionUnderstanding` has no extractable candidates AND no relevant existing memories, Pass 2 and Pass 3 are skipped entirely — a minimal "no memories extracted" summary is written via Python (no LLM calls). Saves two full agent runs on pure-implementation / debugging traces.

In [ ]:
if RUN_PYDANTICAI_THREE_PASS:
    from lerim.agents.extract import (
        build_reflect_agent,
        build_extract_agent as build_extract_pass_agent,
        build_finalize_agent,
        REFLECT_LIMITS,
        EXTRACT_LIMITS,
        FINALIZE_LIMITS,
        _python_finalize_empty,
    )
    from lerim.agents.tools import ExtractDeps
    from lerim.config.providers import build_pydantic_model_from_provider

    tmp_3p, mem_3p = make_temp_memory()
    model_3p = build_pydantic_model_from_provider(
        MODEL_CFG["pai_provider"],
        MODEL_CFG["pai_model"],
    )
    deps_3p = ExtractDeps(memory_root=mem_3p, trace_path=TRACE)

    # Each pass is a separate agent with its own system prompt, tool subset,
    # UsageLimits, and structured output. Running them inline (rather than
    # calling run_extraction_three_pass) lets us inspect each pass's
    # conversation log separately — useful for debugging which pass drops
    # a memory, or where time is spent.
    reflect_agent = build_reflect_agent(model_3p)
    extract_agent_p2 = build_extract_pass_agent(model_3p)
    finalize_agent = build_finalize_agent(model_3p)

    pass_messages: dict[str, list] = {}

    # ── Pass 1: Reflect (read-only) ─────────────────────────────────
    print(f"=== Pass 1 (Reflect) with {MODEL_CFG['pai_model']} ===")
    reflection = await reflect_agent.run(
        "Reflect on the session trace. Read it end-to-end via chunked "
        "read('trace', offset=N, limit=100), scan existing memories, "
        "grep for 'remember', and produce a structured SessionUnderstanding "
        "for the extractor.",
        deps=deps_3p,
        usage_limits=REFLECT_LIMITS,
    )
    understanding = reflection.output
    pass_messages["reflect"] = list(reflection.all_messages())
    print(f"  user_goal: {understanding.user_goal[:100]}")
    print(f"  key_decisions: {len(understanding.key_decisions)}")
    print(f"  important_chunks: {len(understanding.important_chunks)}")
    print(f"  extractable_candidates: {len(understanding.extractable_candidates)}")
    print(f"  existing_memories_relevant: {len(understanding.existing_memories_relevant)}")
    print(f"  messages: {len(pass_messages['reflect'])}")

    # ── Empty-trace short-circuit (no LLM calls in passes 2/3) ────
    if (not understanding.extractable_candidates
            and not understanding.existing_memories_relevant):
        print("\n→ Empty-trace short-circuit: skipping Pass 2 and Pass 3 (no LLM calls)")
        result_3p = _python_finalize_empty(deps_3p, understanding)
    else:
        # ── Pass 2: Extract (read + write + edit) ──────────────────
        print(f"\n=== Pass 2 (Extract) ===")
        extraction = await extract_agent_p2.run(
            "Extract memories per the session understanding below. Apply "
            "extraction criteria and dedup rules. Write novel memories via "
            "write() and edit existing ones via edit() where appropriate.\n\n"
            f"SessionUnderstanding:\n{understanding.model_dump_json(indent=2)}",
            deps=deps_3p,
            usage_limits=EXTRACT_LIMITS,
        )
        extracted = extraction.output
        pass_messages["extract"] = list(extraction.all_messages())
        print(f"  filenames written: {extracted.filenames}")
        print(f"  completion_summary: {extracted.completion_summary[:150]}")
        print(f"  messages: {len(pass_messages['extract'])}")

        # ── Pass 3: Finalize (verify_index + summary) ──────────────
        print(f"\n=== Pass 3 (Finalize) ===")
        finalization = await finalize_agent.run(
            "Finalize the memory index and write a session summary.\n\n"
            f"Session understanding:\n{understanding.model_dump_json(indent=2)}\n\n"
            f"Memories written in this session: {extracted.filenames}",
            deps=deps_3p,
            usage_limits=FINALIZE_LIMITS,
        )
        result_3p = finalization.output
        pass_messages["finalize"] = list(finalization.all_messages())
        print(f"  summary_filename: {result_3p.summary_filename}")
        print(f"  index_ok: {result_3p.index_ok}")
        print(f"  messages: {len(pass_messages['finalize'])}")

    print(f"\nFinal completion summary: {result_3p.completion_summary}")
    report(mem_3p)

    # ── Per-pass message logs (tool calls, returns, text) ──────────
    print(f"\n{'=' * 70}")
    print(f"THREE-PASS CONVERSATION LOG")
    print(f"{'=' * 70}")
    for pass_name in ("reflect", "extract", "finalize"):
        msgs = pass_messages.get(pass_name, [])
        if not msgs:
            print(f"\n--- Pass: {pass_name} — SKIPPED ---")
            continue
        print(f"\n--- Pass: {pass_name} ({len(msgs)} messages) ---")
        for i, msg in enumerate(msgs):
            kind = msg.kind
            print(f"  [{i}] [{kind}]")
            for part in msg.parts:
                ptype = part.part_kind
                if ptype == "system-prompt":
                    print(f"      [system-prompt] ({len(part.content)} chars)")
                elif ptype == "user-prompt":
                    print(f"      [user-prompt] {part.content[:150]}")
                elif ptype == "tool-return":
                    print(f"      [tool-return] {part.tool_name} -> {str(part.content)[:150]}")
                elif ptype == "text":
                    print(f"      [text] {part.content[:200]}")
                elif ptype == "tool-call":
                    print(f"      [tool-call] {part.tool_name}({str(part.args)[:120]})")
                else:
                    print(f"      [{ptype}] {str(part)[:150]}")
else:
    print("SKIPPED (RUN_PYDANTICAI_THREE_PASS=False)")